<a href="https://colab.research.google.com/github/afirdousi/pytorch-basics/blob/main/007_computer_vision_cnn_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
import torch
from torch import nn

import torchvision
from torchvision import datasets
from torchvision import transforms # (Read: https://pytorch.org/vision/stable/transforms.html)
from torchvision.transforms import ToTensor

import matplotlib.pyplot as plt

import pandas as pd
import numpy as np
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using PyTorch version = { torch.__version__ }")
print(f"Using device = { device }")  # We will be doing device agnostic code in this tutorial

Using PyTorch version = 2.0.1+cu118
Using device = cuda


In [48]:
# Intro to Convolutional Neural Networks  (aka as CNN)
# CNNs are also known as ConvNets.
# CNNs are known for their capabilities to find patterns in visual data

# General Structure of a CNN (pretty much same as normal vision model)

# 1 >Input Image
# 2 > Input Layer
# 3 > Pre Processing
# 4 > Convolution Layer (nn.ConvXd()) - Extracts/learns most important features from target images
# 5 > Hidden Activation / Non Linear Activation (nn.ReLU()) - Adds non-linearity to learned features (non-straight lines)
# 6 > Pooling Layer (nn.MaxPool2d()) - Reduces dimentionality of learned image features
# 7 > Output layer/linear layer (nn.Linear) - Takes learned features and outputs them in shape of target labels
# 8 > Output activation (torch.sigmoid) - Converts output logits to prediction probablities

# 4,5,6 can be combined in many different ways and can be repeated many times as a set of conv > relu > pooling

# Deeper CNN (Current research area)
# Having multiple layers of conv > relu > pooling
# Current: The accepted idea is the more layers you add to your CNN, the more deeper pattern it will learn

In [49]:
# Setup Training Data
train_data = datasets.FashionMNIST(
    root = "data", # which folder to download data to
    train = True, # all/most of built in datasets in PyTroch are already divided into train and testing datasets
    download = True, # do want to download?
    transform = torchvision.transforms.ToTensor(), # how do we want to transform the data
    target_transform = None #. how do we want to transform the labels/targets?
)

# Setup Test Data
test_data = datasets.FashionMNIST(
    root = "data",
    train = False,
    download = True,
    transform = torchvision.transforms.ToTensor(),
    target_transform= None
)

In [123]:
# Learn what happens inside CNN on CNN Explainer: https://poloclub.github.io/cnn-explainer/
# Check: https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
# Learn about hyperparameters of CNN on https://poloclub.github.io/cnn-explainer/ (search "Understanding Hyperparameters")

# Let's code
class FashionMNISTModelV2(nn.Module):
  """
  This model architecture will replicate the TinyVGG
  model from CNN explainer website
  """
  # Learn about Orginal VGG: https://medium.com/analytics-vidhya/vggnet-architecture-explained-e5c7318aa5b6

  def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
    super().__init__()
    # In CNN, things are generally referred to as Convolutional Block, think of a convolutional block as a set of conv > relu > pooling
    # deeper CNNs will have multiple conv blocks
    self.conv_block_1 = nn.Sequential(
        nn.Conv2d( in_channels = input_shape, # Conv2d because we are working on a 2D image
                   out_channels = hidden_units,
                   kernel_size = 3, # this could be 3x3 or 3
                   stride = 1,
                   padding = 1), # these are all hyperparametrs used for conv 2d
        nn.ReLU(),
        nn.Conv2d( in_channels = hidden_units,
                   out_channels = hidden_units,
                   kernel_size = 3,
                   stride = 1,
                   padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2)  # takes the max value form a window of n x n witin an image, here n = 2
        # In general, one particular theme in deep neural networks is the images
        # will continue go down smaller and smaller the deeper you go in the network

    )

    self.conv_block_2 = nn.Sequential(
        nn.Conv2d( in_channels = hidden_units,
                   out_channels = hidden_units,
                   kernel_size = 3,
                   stride = 1,
                   padding = 1),
        nn.ReLU(),
        nn.Conv2d( in_channels = hidden_units,
                   out_channels = hidden_units,
                   kernel_size = 3,
                   stride = 1,
                   padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2)
    )

    # finally we will have a classifier layer (where we flatten and classify into target labels)

    # compare this layer to previous simpler FashionMNIST models we made
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features = hidden_units * 4, # there is a trick to calculate this number which we have marked as 0 (more on it later)
                  # initially keep the above *4 to 0 and run to the end, then come here to fix it
                  out_features= output_shape)
    )

  def forward(self, x):
    print('Helloooo in forward...')
    print('Starting conv block 1...')
    x = self.conv_block_1(x)
    print(x.shape)
    print('Starting conv block 2...')
    x = self.conv_block_2(x)
    print(x.shape)
    print('Starting classifier...')
    x = self.classifier(x)
    print('Shape of final output out of classifier layer', x.shape)
    print('Final output',x)

In [120]:
torch.manual_seed(42)

model_1 = FashionMNISTModelV2(input_shape= 1, # since we have black and white images (Vs CNN explainer guys have color videos)
                              hidden_units = 10,
                              output_shape = len(train_data.classes)).to(device)

In [68]:
## Stepping throuhg `nn.Conv2d()` (https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)

# Lets create a dummy data (replicating data in cnn explainer website)
torch.manual_seed(42)
images = torch.randn(size =(32,3,64,64)) # quick note: in cnn explainer, the color channel "3" is shown as last param , 32 here is the batch size
test_image = images[0]

print(f"Image batch shape: {images.shape}")
print(f"Single batch shape: {test_image.shape}")
print(f"Test Image: {test_image}")

Image batch shape: torch.Size([32, 3, 64, 64])
Single batch shape: torch.Size([3, 64, 64])
Test Image: tensor([[[ 1.9269,  1.4873,  0.9007,  ...,  1.8446, -1.1845,  1.3835],
         [ 1.4451,  0.8564,  2.2181,  ...,  0.3399,  0.7200,  0.4114],
         [ 1.9312,  1.0119, -1.4364,  ..., -0.5558,  0.7043,  0.7099],
         ...,
         [-0.5610, -0.4830,  0.4770,  ..., -0.2713, -0.9537, -0.6737],
         [ 0.3076, -0.1277,  0.0366,  ..., -2.0060,  0.2824, -0.8111],
         [-1.5486,  0.0485, -0.7712,  ..., -0.1403,  0.9416, -0.0118]],

        [[-0.5197,  1.8524,  1.8365,  ...,  0.8935, -1.5114, -0.8515],
         [ 2.0818,  1.0677, -1.4277,  ...,  1.6612, -2.6223, -0.4319],
         [-0.1010, -0.4388, -1.9775,  ...,  0.2106,  0.2536, -0.7318],
         ...,
         [ 0.2779,  0.7342, -0.3736,  ..., -0.4601,  0.1815,  0.1850],
         [ 0.7205, -0.2833,  0.0937,  ..., -0.1002, -2.3609,  2.2465],
         [-1.3242, -0.1973,  0.2920,  ...,  0.5409,  0.6940,  1.8563]],

        [[-0.

In [69]:
# Let's give a random image to our model

# Create a single connv2d layer
conv_layer = nn.Conv2d(in_channels= 3, # since we have 3 color channels
                       out_channels= 10,
                       kernel_size=3, #  kernal_size=3 ==  kernal_size=(3x3) # kernal is also known as a filter
                       # kernal is a small convoluion, a small squar that moves left to right to learn patterns in image
                       # go to cnn explainer, click on an image and move/hover the small kernal across the image
                       # Got to CNN explainer and find the heading "Understanding Hyperparameters" and play with padding, kernal size and stride to learn visually
                       stride=1,
                       padding=0
                       )


# Pass the data through the convolution layer
conv_output = conv_layer(test_image)
conv_output

tensor([[[-2.8778e-01, -6.0596e-02, -5.6306e-02,  ...,  2.8654e-01,
           6.6224e-01, -2.3216e-01],
         [-9.8911e-01, -4.0099e-01,  4.1832e-01,  ...,  4.7459e-01,
          -1.8552e-01, -5.7622e-01],
         [-4.1340e-02, -2.3277e-01,  3.7418e-01,  ...,  2.8255e-02,
           1.4923e-01,  1.4236e-01],
         ...,
         [-8.0374e-01, -7.6687e-01, -5.9457e-02,  ...,  1.7452e-01,
           4.2594e-01, -4.8341e-01],
         [-1.4512e-01, -1.1566e-01,  6.1783e-01,  ...,  2.4126e-01,
          -3.6626e-01,  3.5645e-01],
         [ 3.6096e-02,  1.5214e-01,  2.3123e-01,  ...,  3.0904e-01,
          -4.9680e-01, -7.2258e-01]],

        [[-1.0853e+00, -1.6079e+00,  1.3346e-01,  ...,  2.1698e-01,
          -1.7643e+00,  2.5263e-01],
         [-8.2507e-01,  6.3866e-01,  1.8845e-01,  ..., -1.0936e-01,
           4.8068e-01,  8.4869e-01],
         [ 6.4927e-01, -4.2061e-03, -4.9991e-01,  ...,  5.8356e-01,
           2.4611e-01,  6.6233e-01],
         ...,
         [ 9.8860e-02,  1

In [70]:
conv_output.shape # [10, 62, 62] means our image has gone down from [ 3,64,64 ] to [ 10, 62, 62]

torch.Size([10, 62, 62])

In [71]:
# You can now play around with kernal and other hyperparameters to see how it changes the output layer
# For example:

conv_layer = nn.Conv2d(in_channels= 3,
                       out_channels= 10,
                       kernel_size=2,
                       stride=2,
                       padding=1
                       )


# Pass the data through the convolution layer
conv_output = conv_layer(test_image)
conv_output.shape

torch.Size([10, 33, 33])

In [72]:
# another example (play with more changing one or more variables at a time)
conv_layer = nn.Conv2d(in_channels= 3,
                       out_channels= 10,
                       kernel_size=3,
                       stride=3,
                       padding=1
                       )

conv_output = conv_layer(test_image)
conv_output.shape

torch.Size([10, 22, 22])

In [73]:
# Stepping Through nn.MaxPool2d() (Read More: https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html)
# Overall these exersized are meant to show you how you debug your ML code step by step

test_image.shape

torch.Size([3, 64, 64])

In [74]:
# Print out original image shape without unsqueezed dimension

print(f"Test image original shape: {test_image.shape}")
print(f"Test image with unsqueezed dimension: {test_image.unsqueeze(0).shape}")

# Create a sampple nn.MaxPool2d layer
max_pool_layer = nn.MaxPool2d(kernel_size= 2)
# First, lets pass data through a conv layer (generally this is what you would do in real model too, pass through Conv2d and then pass through MaxPool2d)
test_image_through_conv = conv_layer(test_image.unsqueeze(dim= 0))
print(f"Shape after going through conv_layer(): {test_image_through_conv.shape} ")

# Pass data through max pool layer
test_image_through_conv_and_max_pool = max_pool_layer(test_image_through_conv)
print(f"Shape after going through conv_layer() and max pool layer: {test_image_through_conv_and_max_pool.shape} ")

# Exercise: Change the kernal size here and see the impact

Test image original shape: torch.Size([3, 64, 64])
Test image with unsqueezed dimension: torch.Size([1, 3, 64, 64])
Shape after going through conv_layer(): torch.Size([1, 10, 22, 22]) 
Shape after going through conv_layer() and max pool layer: torch.Size([1, 10, 11, 11]) 


In [75]:
# Observe how each layer decreases the size of the image

# We are going from a higher dimension space to a lower dimension space / smaller vector space in terms of dimensionality of a tensor
# Still, this small dimension space represents the original data

# The idea is once we have learned certain patterns from convolutional layer,
# we can take max of the most important parts of the image and will still learn the overall image patterns

# reduction in dimension is the intelligence here

In [76]:
torch.manual_seed(42)

# Create a random tensor with a similar number of dimension to our image

random_tensor = torch.randn(size=(1,1,2,2))
random_tensor

print(f"Random Tensor: \n{ random_tensor }")
print(f"Random Tensor Shape: \n{ random_tensor.shape }")

Random Tensor: 
tensor([[[[0.3367, 0.1288],
          [0.2345, 0.2303]]]])
Random Tensor Shape: 
torch.Size([1, 1, 2, 2])


In [77]:


# Create a nax pool layer
max_pool_layer = nn.MaxPool2d(kernel_size= 2)

# Pass random_tensor through max pool layer
max_pool_tensor = max_pool_layer(random_tensor)

print(f"Max Pool Tensor: \n { max_pool_tensor }")
print(f"Max Pool Tensor Shape: \n { max_pool_tensor.shape }")

Max Pool Tensor: 
 tensor([[[[0.3367]]]])
Max Pool Tensor Shape: 
 torch.Size([1, 1, 1, 1])


In [78]:
# Observe how we reduced the shape from 2 x 2 to 1 x 1 using Max Pool layer which picks up the max of the entire 2x2 tensor
# Convolutional network is tyring to learn the most important parts of the image

# Exercise: Now take a dummy image and pass it through the entire tiny VGG net that we have built above called FashionMNISTModelV2

In [79]:
# A common practice of learning in ML is you take a model, replicate it (implement it yourself) and then pass your own data to test it

In [91]:
# So far we are working towards converting our FashionMNIST ---to---> CNN

# Lets create a dummy tensor and pass it through our model

rand_image_tensor = torch.rand(size=(1,10,10))
rand_image_tensor

# Check model_1 to see its layers
model_1

FashionMNISTModelV2(
  (conv_block_1): Sequential(
    (0): Conv2d(1, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block_2): Sequential(
    (0): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=0, out_features=10, bias=True)
  )
)

In [107]:
# Now we want to pass our random image tensor to model 1 which has conv block 1 and conb block 2
# conv 1 takes 1 and outputs 10

# how do convert our rand_image_tensor is now 1 dimension. lets pass it
rand_image_tensor.flatten()

tensor([0.9360, 0.4471, 0.8521, 0.6757, 0.6427, 0.1812, 0.9759, 0.7290, 0.5773,
        0.8537, 0.0704, 0.6077, 0.2646, 0.4492, 0.6638, 0.7214, 0.6147, 0.1130,
        0.1871, 0.5087, 0.4134, 0.0815, 0.9926, 0.5629, 0.1447, 0.7193, 0.7172,
        0.2547, 0.7507, 0.1898, 0.6732, 0.9750, 0.8301, 0.7468, 0.3983, 0.5939,
        0.6745, 0.6221, 0.3987, 0.5698, 0.2975, 0.2174, 0.0386, 0.0923, 0.1861,
        0.9779, 0.6001, 0.0477, 0.2444, 0.6168, 0.3503, 0.3585, 0.3122, 0.3779,
        0.5575, 0.5296, 0.8212, 0.8200, 0.9115, 0.4841, 0.2767, 0.5017, 0.1689,
        0.7046, 0.4397, 0.3399, 0.0761, 0.7410, 0.7791, 0.9561, 0.3229, 0.6507,
        0.5188, 0.1815, 0.1874, 0.7560, 0.2128, 0.8641, 0.6747, 0.6708, 0.8293,
        0.7447, 0.7156, 0.7594, 0.2878, 0.2349, 0.4084, 0.7776, 0.8499, 0.8682,
        0.3423, 0.4366, 0.1566, 0.8083, 0.1684, 0.2943, 0.9129, 0.6215, 0.2907,
        0.1633])

In [122]:
model_1(rand_image_tensor.unsqueeze(0).to(device)) # one pass through our model

Helloooo in forward...
Starting conv block 1...
torch.Size([1, 10, 5, 5])
Starting conv block 2...
torch.Size([1, 10, 2, 2])
Starting classifier...
Shape of final output out of classifier layer torch.Size([1, 10])
Final output tensor([[-0.0261, -0.0434,  0.0197,  0.1462, -0.0393,  0.1240,  0.0723,  0.0726,
         -0.1243, -0.0829]], device='cuda:0', grad_fn=<AddmmBackward0>)


In [ ]:
# lets fix the above error